# Tugas 4: Implementasi Blockchain Sederhana dengan Python
**Mata Kuliah:** IF219 Teknologi Blockchain  
**Program Studi:** S1 Teknologi Informasi  
**Institut Teknologi Tangerang Selatan (ITTS)**  

---
**Identitas Mahasiswa:**
- **Nama:** Rafael Bramantyo Buana Putra
- **NIM:** 1002240003
- **Pertemuan:** 6 (Tugas Individu - Bobot 5%)
- **Sub-CPMK:** SUB-IF219-2-3 Menerapkan mekanisme pembentukan blok, transaksi, dan validasi rantai pada program blockchain sederhana.

---
## Panduan Menjalankan Notebook (Google Colab / Jupyter)
1. Jalankan setiap cell secara berurutan dari atas ke bawah (`Shift + Enter`).
2. Program hanya memerlukan library bawaan Python (`hashlib`, `json`, `time`), tidak ada package eksternal yang perlu diinstall.
3. Di bagian akhir terdapat pengujian otomatis 5 skenario sesuai rubrik tugas termasuk pengujian 2 varian tampering `previous_hash`.

## 1. Definisi Class Block dan Blockchain

In [ ]:
import hashlib
import json
import time

class Block:
    """
    Struktur data Block merepresentasikan sebuah blok dalam rantai blockchain.
    """
    def __init__(self, index, data, previous_hash, nonce=0):
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        """Menghitung hash SHA-256 dari seluruh isi blok secara deterministik."""
        payload = {
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }
        # sort_keys=True menjamin representasi JSON konsisten dan deterministik
        raw = json.dumps(payload, sort_keys=True).encode()
        return hashlib.sha256(raw).hexdigest()

    def __repr__(self):
        return f"Block(Index={self.index}, Hash={self.hash[:16]}..., PrevHash={self.previous_hash[:16]}...)"


class Blockchain:
    """
    Struktur data Blockchain yang mengelola rantai blok dan validasi integritas ledger.
    """
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        """Membuat genesis block (blok ke-0) secara otomatis."""
        return Block(
            index=0,
            data={"keterangan": "Genesis Block ITTS Blockchain"},
            previous_hash="0" * 64
        )

    def latest_block(self):
        """Mengambil blok terakhir dalam rantai."""
        return self.chain[-1]

    def add_block(self, data):
        """Menambahkan blok baru dengan previous_hash dari blok terakhir."""
        prev_block = self.latest_block()
        new_block = Block(
            index=len(self.chain),
            data=data,
            previous_hash=prev_block.hash
        )
        self.chain.append(new_block)
        return new_block

    def is_valid(self):
        """
        Validasi integritas seluruh blockchain:
        1. Integritas genesis block.
        2. Integritas hash internal setiap blok (hash == calculate_hash).
        3. Keterikatan relasi previous_hash (current.previous_hash == previous.hash).
        """
        # Verifikasi genesis block
        genesis = self.chain[0]
        if genesis.index != 0 or genesis.previous_hash != "0" * 64:
            return False
        if genesis.hash != genesis.calculate_hash():
            return False

        # Verifikasi blok-blok berikutnya
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # 1. Cek apakah isi blok dimanipulasi (integritas internal)
            if current.hash != current.calculate_hash():
                return False

            # 2. Cek apakah keterikatan rantai terputus (keterikatan blok sebelumnya)
            if current.previous_hash != previous.hash:
                return False

        return True

print("Class Block dan Blockchain berhasil diinisialisasi.")

## 2. Pengujian Skenario 1 & 2
- **Skenario 1:** Chain baru hanya memiliki genesis block secara otomatis dan bernilai valid.
- **Skenario 2:** Tambahkan minimal 3 blok dengan transaksi berbeda dan verifikasi keabsahan rantai.

In [ ]:
# Inisialisasi Blockchain
bc = Blockchain()

print("=== [SKENARIO 1] Inisialisasi Chain & Genesis Block ===")
print(f"Jumlah Blok Saat Ini: {len(bc.chain)}")
print(f"Genesis Block Index : {bc.chain[0].index}")
print(f"Genesis Prev Hash   : {bc.chain[0].previous_hash}")
print(f"Genesis Hash        : {bc.chain[0].hash}")
print(f"Validitas Chain     : {bc.is_valid()} (Expected: True)")

print("\n=== [SKENARIO 2] Tambah Minimal 3 Blok Transaksi ===")
t1 = {"pengirim": "Alice", "penerima": "Bob", "nominal": 50, "token": "IDR"}
t2 = {"pengirim": "Bob", "penerima": "Charlie", "nominal": 20, "token": "IDR"}
t3 = {"pengirim": "Charlie", "penerima": "Dave", "nominal": 15, "token": "IDR"}

b1 = bc.add_block(t1)
b2 = bc.add_block(t2)
b3 = bc.add_block(t3)

print(f"Blok 1 Ditambahkan: Hash={b1.hash[:24]}...")
print(f"Blok 2 Ditambahkan: Hash={b2.hash[:24]}...")
print(f"Blok 3 Ditambahkan: Hash={b3.hash[:24]}...")
print(f"Total Blok Sekarang: {len(bc.chain)}")
print(f"Validitas Chain    : {bc.is_valid()} (Expected: True)")

## 3. Pengujian Skenario 3 & 4 (Eksperimen Tampering)
- **Skenario 3:** Ubah data transaksi pada blok tengah (Block Index 2) tanpa menghitung ulang chain.
- **Skenario 4 (Dua Varian):**
  - **Varian 4A:** `previous_hash` diubah TANPA menghitung ulang hash blok (Harus `False` karena hash internal tidak valid).
  - **Varian 4B:** `previous_hash` diubah DENGAN menghitung ulang hash blok (Harus `False` karena keterikatan dengan blok sebelumnya putus).

In [ ]:
print("=== [SKENARIO 3] Tampering Data pada Blok Tengah (Index 2) ===")
data_asli = bc.chain[2].data
print(f"Data Asli Blok 2: {data_asli}")

# Manipulasi data secara ilegal
bc.chain[2].data = {"pengirim": "Bob", "penerima": "Attacker", "nominal": 9999, "token": "IDR"}
print(f"Data Palsu Blok 2: {bc.chain[2].data}")
print(f"Validitas Chain setelah Tampering Data: {bc.is_valid()} (Expected: False)")

# Kembalikan data asli dan pulihkan chain
bc.chain[2].data = data_asli
print(f"Validitas Chain setelah Data Dipulihkan: {bc.is_valid()} (Expected: True)")

print("\n=== [SKENARIO 4] Dua Varian Tampering previous_hash pada Blok Index 2 ===")
original_prev_hash = bc.chain[2].previous_hash
original_hash = bc.chain[2].hash
fake_prev_hash = "deadbeef" * 8

# Varian 4A: previous_hash diubah TANPA menghitung ulang hash blok
print("--- Skenario 4A: previous_hash diubah TANPA menghitung ulang hash blok ---")
bc.chain[2].previous_hash = fake_prev_hash
print(f"Previous Hash Dimanipulasi : {bc.chain[2].previous_hash}")
print(f"Hash Blok Tidak Diperbarui : {bc.chain[2].hash}")
print(f"Validitas Chain (4A): {bc.is_valid()} (Expected: False)")

# Pulihkan chain setelah varian 4A
bc.chain[2].previous_hash = original_prev_hash
bc.chain[2].hash = original_hash
print(f"Validitas setelah Pemulihan 4A: {bc.is_valid()} (Expected: True)")

# Varian 4B: previous_hash diubah DENGAN menghitung ulang hash blok
print("\n--- Skenario 4B: previous_hash diubah DENGAN menghitung ulang hash blok ---")
bc.chain[2].previous_hash = fake_prev_hash
bc.chain[2].hash = bc.chain[2].calculate_hash() # Hash internal blok 2 konsisten
print(f"Previous Hash Dimanipulasi : {bc.chain[2].previous_hash}")
print(f"Hash Dihitung Ulang        : {bc.chain[2].hash}")
print(f"Validitas Chain (4B): {bc.is_valid()} (Expected: False)")

# Pulihkan chain setelah varian 4B
bc.chain[2].previous_hash = original_prev_hash
bc.chain[2].hash = original_hash
print(f"Validitas setelah Pemulihan 4B: {bc.is_valid()} (Expected: True)")

## 4. Pengujian Skenario 5: Cetak Rincian Seluruh Blok
Mencetak `index`, `timestamp`, `data`, `previous_hash`, dan `hash` setiap blok untuk mengamati keterikatan kriptografis rantai.

In [ ]:
print("=== [SKENARIO 5] Rincian Struktur Seluruh Blok Blockchain ===\n")
print("=" * 75)
for block in bc.chain:
    print(f"Index         : {block.index}")
    print(f"Timestamp     : {block.timestamp}")
    print(f"Data          : {block.data}")
    print(f"Previous Hash : {block.previous_hash}")
    print(f"Current Hash  : {block.hash}")
    print("-" * 75)

## 5. Analisis Ilmiah: Mengapa Tampering Dapat Dideteksi?
Integritas ledger blockchain dijamin melalui 4 fondasi utama:
1. **Hash Deterministik dari Isi Blok:** Seluruh atribut blok diubah menjadi format representasi konsisten menggunakan `json.dumps(..., sort_keys=True)` sebelum dihitung dengan fungsi hash. Setiap input yang sama selalu menghasilkan output digest yang persis sama secara deterministik.
2. **Avalanche Effect (Efek Longsoran Salju):** Sifat dasar algoritma SHA-256 memastikan perubahan sekecil apa pun (bahkan satu bit atau karakter) pada data transaksi akan mengubah nilai hash total secara drastis tak terduga.
3. **Audit Konsistensi Internal (`is_valid`):** Fungsi `is_valid()` menghitung ulang hash dari seluruh muatan blok dan membandingkannya langsung dengan hash yang tersimpan (`current.hash == current.calculate_hash()`). Modifikasi data tanpa mengubah hash blok langsung terdeteksi pada tahap ini (Skenario 3 & Skenario 4A).
4. **Keterikatan Kriptografis Rantai (`previous_hash`):** Atribut `previous_hash` mengikat setiap blok secara kaku ke blok pendahulunya. Jika penyerang memperbarui hash internal suatu blok agar konsisten secara internal, nilai hash blok tersebut tidak lagi cocok dengan `previous_hash` yang disimpan oleh blok setelahnya (`current.previous_hash == previous.hash`). Dengan demikian, memodifikasi satu blok akan memutus keabsahan rantai seluruh blok setelahnya (Skenario 4B).